### 0. Preliminaries
Import necessary packages

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

### 1. Model Setup
Define constant parameters, user functions, and simulation individuals

In [6]:
# Define constant parameters
county_size = 20 # miles
taxi_speed = 20 # miles per hour
base_fare = 3.00 # pounds
per_mile_fare = 2.00 # pounds per mile
petrol_cost = 0.2 # pounds per mile

In [7]:
def distance(loc1, loc2):
    return np.sqrt((loc1[0] - loc2[0])**2 + (loc1[1] - loc2[1])**2)

def generate_location():
    return (random.uniform(0, county_size), random.uniform(0, county_size))

def travel_time(d_OD):
    mu = d_OD / taxi_speed
    return random.uniform(0.8*mu, 1.2*mu) 

### 2. Run Simulation (with improvements)

In [10]:
def run_simulation(T, iter_D , iter_R, lambda1, lambda2):
    ''' Runs the rideshare Simulation with BoxCar's initial assumptions. 
            Inputs:
            - T: Termination Time (in hours)
            - iter_D: Driver ID iteration start
            - iter_R: Rider ID iteration start
            Output:
            - results: dictionary of performance metrics indexed by metric type '''
    # Initialise performance statistics
    N_D = 0
    N_R = 0
    T_D = 0
    T_R = 0
    P = 0
    R = 0    # total completed rides
    matched_rides = 0    # started but dropoff is after TerminationTime - do not count driver earnings
    picked_up_rides = 0    # matched and actually gets picked up before TerminationTime
    abandoned_rides = 0    # abandoned due to patience time being reached
    T_P = []
    T_All = []
    T_W = []

    driver_ride_counts = {}
    driver_idle_time = {}
    driver_last_busy_time = {}
    
    # initialize simulation
    TNOW = 0
    TerminationTime = T #hrs
    EventCalendar = []
    def schedule_event(time, event_type, data=None):
        '''Event scheduler that keeps track of times, event type, and user data (optional)'''
        EventCalendar.append((time,event_type,data))
   
    drivers =[]
    idle_drivers=[]
    riders = []
    waiting_riders=[]

    driver_id_counter = iter_D
    rider_id_counter = iter_R

    schedule_event(random.expovariate(3), 1)   # 1 = Driver Arrival
    schedule_event(random.expovariate(30), 2)  # 2 = Rider Arrival
    schedule_event(TerminationTime, 8)             # 8 = Termination

    ### Weighted matching (calculate score for driver)
    def driver_score(driver, rider, TNOW, lambda1, lambda2):
        
        d = distance(driver['location'], rider['origin'])

        time_active = TNOW - driver['online_time'] + 1e-6
        profit_rate = driver['earnings'] / time_active

        break_time = TNOW - driver_last_busy_time[driver['id']]

        score = d + lambda1 * profit_rate - lambda2 * break_time

        return score
    
    while TNOW < TerminationTime:
        TNEXT, TypeNext, Data = min(EventCalendar, key=lambda x: x[0])
        EventCalendar.remove((TNEXT, TypeNext, Data))

        ## Update Performance Stats Here (Alice)

        TNOW = TNEXT

        # DriverArrival
        if TypeNext == 1:
            driver_id_counter += 1

            # Generate driver location and offline time (initialize other attributes)
            driver = {
                "id": driver_id_counter,
                "location": generate_location(),
                'offline_time': TNOW + random.uniform(5,8),
                'idle': True,
                'active': True,
                'earnings': 0,
                'online_time': TNOW
            }
            drivers.append(driver)

            # Update performance metrics immediately
            N_D += 1
            driver['online_time'] = TNOW
            driver_ride_counts[driver['id']] = 0
            driver_idle_time[driver['id']] = 0
            driver_last_busy_time[driver['id']] = TNOW

            
            # Schedule offline time
            schedule_event(driver['offline_time'], event_type=7, data = driver)

            # Match driver to waiting rider
            if waiting_riders:
                # match closest waiting rider
                rider = min(
                    waiting_riders,
                    key=lambda r: driver_score(driver, r, TNOW, lambda1, lambda2)
                )
                waiting_riders.remove(rider)
                
                # update statuses
                rider['status'] = 'matched'
                driver['idle'] = False
                
                # schedule pickup start
                schedule_event(TNOW, event_type=3, data = (driver, rider)) 

                # update performance metrics
                matched_rides += 1
            else:
                idle_drivers.append(driver)
            
            # Schedule next driver arrival
            schedule_event(TNOW + random.expovariate(3), event_type=1)

        # RiderArrival
        elif TypeNext == 2: 
            rider_id_counter += 1

            # Generate rider location and patience time (initialize other attributes)
            rider = {
                "id": rider_id_counter,
                "origin": generate_location(),
                'destination': generate_location(),
                'patience_time': random.expovariate(5),
                'status': 'waiting'
            }
            riders.append(rider)
            
            # Match driver to idle active driver
            idle_active_drivers = [driver for driver in idle_drivers if driver['active']]

            if idle_active_drivers:
                # Use weighted matching metric
                driver = min(idle_active_drivers,
                    key=lambda d: driver_score(d, rider, TNOW, lambda1, lambda2)
                )
                # update statuses
                rider['status'] = 'matched'
                driver['idle'] = False

                # schedule PickupStart
                schedule_event(TNOW, event_type=3, data = (driver,rider))

                # update performance metrics
                matched_rides += 1
            else:
                waiting_riders.append(rider)

                #schedule patience deadline for waiting rider
                schedule_event(TNOW+rider['patience_time'], event_type=6, data = rider)
            
            # schedule next rider arrival
            schedule_event(TNOW + random.expovariate(30), event_type=2)

            # Update performance metrics
            N_R += 1
            rider['arrival_time'] = TNOW
            #T_P.append(rider['patience_time'])

        # PickupStart
        elif TypeNext == 3:
            driver,rider = Data

            # in very rare event that driver becomes offline at exact time it gets matched, 
            # add rider back to waiting list
            if not driver['active']:
                rider['status'] = 'waiting'
                waiting_riders.append(rider)
                continue
            
            # Calculate actual travel time 
            d_P = distance(driver['location'],rider['origin'])
            t_P = travel_time(d_P)

            # Schedule Pickup event
            schedule_event(TNOW + t_P, event_type=4, data = (driver,rider,d_P))

            # Update performance metrics
            waiting_time = TNOW - rider['arrival_time']  # time waited between rider arrival and matching
            T_W.append(waiting_time)
            T_All.append(waiting_time)
            T_P.append(rider['patience_time'])
            driver_ride_counts[driver['id']] += 1
            driver_idle_time[driver['id']] += TNOW - driver_last_busy_time[driver['id']]
            driver_last_busy_time[driver['id']] = TNOW
            
        
        # Pickup
        elif TypeNext == 4:
            driver, rider, d_P = Data
            
            # Calculate actual travel time
            d_OD = distance(rider['origin'], rider['destination'])
            t_OD = travel_time(d_OD)

            # Schedule Dropoff
            schedule_event(TNOW + t_OD, event_type=5, data = (driver,rider,d_P,d_OD,t_OD))

            # update performance metrics
            picked_up_rides += 1

        # Dropoff
        elif TypeNext == 5:
            driver, rider, d_P, d_OD,t_OD = Data

            # update rider status
            rider["status"] = "dropped-off"

            # calculate driver earnings
            rider_fare = base_fare + per_mile_fare * d_OD
            driver_cost = petrol_cost*(d_P+d_OD)
            driver_profit = rider_fare - driver_cost
            driver['earnings'] += driver_profit      # changed this to make it cumulative profit
            
            # Update driver information
            driver['location'] = rider['destination']
            driver['idle'] = True

            # Check if driver is meant to be offline now
            if TNOW >= driver['offline_time']:
                driver['active'] = False
            else:
                if waiting_riders:
                    next_rider = min(
                        waiting_riders,
                        key=lambda r: driver_score(driver, r, TNOW, lambda1, lambda2)
                    )
                    waiting_riders.remove(next_rider)
                    next_rider['status'] = 'matched'
                    driver['idle'] = False
                    # schedule next pickup if driver gets matched to a new rider
                    schedule_event(TNOW, event_type=3, data = (driver, next_rider))

                    # update performance metrics
                    matched_rides += 1
                else:
                    idle_drivers.append(driver)

            # Update performance metrics
            R += 1
            T_R += t_OD  
            P += driver_profit
            driver_last_busy_time[driver['id']] = TNOW
        
        # RiderAbandonment
        elif TypeNext == 6:
            rider = Data

            # remove abandonded riders and update status
            if rider['status'] == 'waiting':
                waiting_riders.remove(rider)
                rider["status"] = "abandoned"
                abandoned_rides += 1

                # Update performance metrics
                T_All.append(rider['patience_time'])
                T_P.append(rider['patience_time'])
        
        # DriverOffline
        elif TypeNext == 7:
            driver = Data
            
            # remove offline drivers and update status
            if driver["active"] and driver["idle"]:
                driver["active"] = False
                if driver in idle_drivers:
                    idle_drivers.remove(driver)

            # Update performance metrics
            T_D += TNOW - driver['online_time']
            driver_idle_time[driver['id']] += TNOW - driver_last_busy_time[driver['id']]

        else:
            # Update performance metrics
            for driver in drivers:
                if driver['active']:
                    T_D += TNOW - driver['online_time']
                    
                    if driver['idle']:
                        driver_idle_time[driver['id']] += TNOW - driver_last_busy_time[driver['id']]

    
    # Individual driver metrics
    # driver_profits = []
    # driver_rides = []
    # driver_idle = []
    # driver_online = []
    # driver_idle_per_ride = []
    # driver_working_ratio = []

    # for driver in drivers:
    #     d_ID = driver['id']
    #     online_time = 0
    #     idle_time = 0
    #     rides = driver_ride_counts.get(d_ID, 0)
        
    #     # compute idle time
    #     if d_ID in driver_idle_time:
    #         idle_time = driver_idle_time[d_ID]
        
    #     # compute online time
    #     if 'online_time' in driver:
    #         if driver['active']:
    #             online_time = TerminationTime - driver['online_time']
    #         else:
    #             online_time = driver['offline_time'] - driver['online_time']
        
    #     profit = driver['earnings']
        
    #     driver_profits.append(profit)
    #     driver_rides.append(rides)
    #     driver_idle.append(idle_time)
    #     driver_online.append(online_time)
        
        # if rides > 0:
        #     driver_idle_per_ride.append(idle_time / rides)
        # else:
        #     driver_idle_per_ride.append(0)
        
        # if online_time > 0:
        #     driver_working_ratio.append((online_time - idle_time) / online_time)
        # else:
        #     driver_working_ratio.append(0)
    
    results = {
        "system_metrics": {
            "N_D": N_D,
            "N_R": N_R,
            "R": R,
            "matched_rides": matched_rides,
            "picked_up_rides": picked_up_rides,
            "abandoned_rides": abandoned_rides
        },
    
        "financial_metrics": {
            "P": P
        },

        "time_metrics": {
            "T_D": T_D,
            "T_R": T_R,
            "T_W": T_W,
            "T_P": T_P,
            "T_All": T_All
        },

        "driver_metrics": {
            # "driver_profits": driver_profits,
            # "driver_rides": driver_rides,
            # "driver_idle": driver_idle,
            # "driver_online": driver_online,
            # "driver_idle_per_ride": driver_idle_per_ride,
            # "driver_working_ratio": driver_working_ratio,
            "driver_ride_counts": driver_ride_counts,
            "driver_idle_time": driver_idle_time,
            "driver_last_busy_time": driver_last_busy_time
        },

        "entities": {
            "drivers": drivers,
            "riders": riders
        }
    }

    return results


### Grid search for $\lambda_1$ and $\lambda_2$

In [11]:
lambda1_values = [0, 0.1, 0.5, 1]
lambda2_values = [0, 0.1, 0.5, 1]

iteration_results = []

for l1 in lambda1_values:
    for l2 in lambda2_values:

        sim = run_simulation(T=100, iter_D=0, iter_R=0, lambda1=l1, lambda2=l2)

        iteration_results.append({
            "lambda1": l1,
            "lambda2": l2,
            "avg_wait": np.mean(sim["time_metrics"]["T_W"]),
            "abandon_rate": sim["system_metrics"]["abandoned_rides"] / sim["system_metrics"]["N_R"],
            "driver_profit": sim["financial_metrics"]["P"]
        })

In [12]:
iteration_results

[{'lambda1': 0,
  'lambda2': 0,
  'avg_wait': np.float64(0.0029154967727911246),
  'abandon_rate': 0.03233411923206467,
  'driver_profit': np.float64(60795.66144757693)},
 {'lambda1': 0,
  'lambda2': 0.1,
  'avg_wait': np.float64(1.555372862442713e-05),
  'abandon_rate': 0.0,
  'driver_profit': np.float64(62460.96495949568)},
 {'lambda1': 0,
  'lambda2': 0.5,
  'avg_wait': np.float64(0.0008391108686429993),
  'abandon_rate': 0.016850291639662993,
  'driver_profit': np.float64(64231.66998402255)},
 {'lambda1': 0,
  'lambda2': 1,
  'avg_wait': np.float64(0.001055954229243441),
  'abandon_rate': 0.030774214447683836,
  'driver_profit': np.float64(63028.69932827556)},
 {'lambda1': 0.1,
  'lambda2': 0,
  'avg_wait': np.float64(0.002166345811138476),
  'abandon_rate': 0.039600665557404324,
  'driver_profit': np.float64(61207.719565331376)},
 {'lambda1': 0.1,
  'lambda2': 0.1,
  'avg_wait': np.float64(0.0),
  'abandon_rate': 0.0,
  'driver_profit': np.float64(65159.60857285429)},
 {'lambda1':

## combine results from multiple simulations

In [2]:
TerminationTime = 100
num_sims = 10

results = []

next_driver_id = 0
next_rider_id = 0

for i in range(num_sims):

    sim_result = run_simulation(
        TerminationTime,
        iter_D=next_driver_id,
        iter_R=next_rider_id
    )

    results.append(sim_result)

    # Update counters for next simulation
    next_driver_id += sim_result["system_metrics"]["N_D"]
    next_rider_id += sim_result["system_metrics"]["N_R"]

def combine_results(results):
    final = {
        # Tally each metric from each run of the simulation
        "system_metrics": {
            k: sum(r["system_metrics"][k] for r in results)
            for k in results[0]["system_metrics"]
        },

        "financial_metrics": {
            "P": sum(r["financial_metrics"]["P"] for r in results)
        },

        "time_metrics": {
            "T_D": sum(r["time_metrics"]["T_D"] for r in results),
            "T_R": sum(r["time_metrics"]["T_R"] for r in results),
            # Aggregate the lists
            "T_W": [x for r in results for x in r["time_metrics"]["T_W"]],
            "T_P": [x for r in results for x in r["time_metrics"]["T_P"]],
            "T_All": [x for r in results for x in r["time_metrics"]["T_All"]],
        },

        "driver_metrics": {
            # set up driver-id based dictionaries for each of these counts
            "driver_ride_counts": {
                k: v
                for r in results
                for k, v in r["driver_metrics"]["driver_ride_counts"].items()
            },

            "driver_idle_time": {
                k: v
                for r in results
                for k, v in r["driver_metrics"]["driver_idle_time"].items()
            },

            "driver_last_busy_time": {
                k: v
                for r in results
                for k, v in r["driver_metrics"]["driver_last_busy_time"].items()
            },
        },

        "entities": {
            # aggregate the lists of dictionaries for all drivers and riders
            "drivers": [d for r in results for d in r["entities"]["drivers"]],
            "riders": [d for r in results for d in r["entities"]["riders"]],
        }
    }

    return final


final = combine_results(results)

TypeError: run_simulation() missing 2 required positional arguments: 'lambda1' and 'lambda2'